In [1]:
# !pip install jsonschema
import json
from jsonschema import validate
from constants import ValidationError, ErrorType, assert_all_logs
from engine import Engine
import os

data_file = 'data.json'
schema_file = 'schema.json'
data_enum_error_file = 'data_enum_error.json'
data_type_error_file = 'data_type_error.json'
data_no_schema_error_file = 'data_no_schema_error.json'
data_unclosed_error_file = 'data_unclosed_error.json'

In [2]:
json_data = json.load(open(data_file))
json_schema = json.load(open(schema_file))

# Validate the JSON data against the schema using jsonschema library
# jsonschema requires valid python dict for both data and schema
validate(instance=json_data, schema=json_schema)

In [3]:
def verify_json(schema=schema_file, data=data_file, max_depth = None):
    engine = Engine(schema=schema, target=data, max_depth=max_depth)
    log = engine.run()
    return log

In [4]:
# valid json data verification
verify_json(data=data_file)

push: top_object
schema ids from parent: [ref(0)]
my schema ids: [ref(0)]
additional: True
push: top_object.a
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(2)]
pop: top_object.a
my states: []
my schema ids: [ref(2)]
children schema_ids: []
verifiy node: schema2
validation errors: {'type': <NO ERROR>: valid}
push: top_object.b
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(3)]
additional: False
push: top_object.b.c
schema ids from parent: [{'c': ref(5), None: ref(-2)}]
my schema ids: [ref(5)]
pop: top_object.b.c
my states: []
my schema ids: [ref(5)]
children schema_ids: []
verifiy node: schema5
validation errors: {'type': <NO ERROR>: valid}
pop: top_object.b
my states: []
my schema ids: [ref(3)]
children schema_ids: [{'c': ref(5), None: ref(-2)}]
verifiy node: schema3
start verifying: [[    <NO ERROR>: valid
]]
all states: [[    <NO ERROR>: valid
]], state idx: 0
validate all: [F

In [5]:
# type error verification
log = verify_json(data=data_type_error_file)
target_errors = {
    'top_object.a': ValidationError(ErrorType.BAD_VALUE, {'value': 'c', 'rule': 'type(number)'}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'a'})
}

# print(log)
# assert_all_logs(log, target_errors)
# log

push: top_object
schema ids from parent: [ref(0)]
my schema ids: [ref(0)]
additional: True
push: top_object.a
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(2)]
pop: top_object.a
my states: []
my schema ids: [ref(2)]
children schema_ids: []
verifiy node: schema2
fail, add error: <BAD VALUE>: value(c) violates schema[type(number)]
validation errors: {'type': <BAD VALUE>: value(c) violates schema[type(number)]}
push: top_object.b
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(3)]
additional: False
push: top_object.b.c
schema ids from parent: [{'c': ref(5), None: ref(-2)}]
my schema ids: [ref(5)]
pop: top_object.b.c
my states: []
my schema ids: [ref(5)]
children schema_ids: []
verifiy node: schema5
validation errors: {'type': <NO ERROR>: valid}
pop: top_object.b
my states: []
my schema ids: [ref(3)]
children schema_ids: [{'c': ref(5), None: ref(-2)}]
verifiy node: schema3
start veri

In [6]:
# unclosed error verification
log = verify_json(data=data_unclosed_error_file)

target_errors = {
    'top_object.d': ValidationError(ErrorType.UNCLOSED),
    'top_object': ValidationError(ErrorType.UNCLOSED)
}

# assert_all_logs(log, target_errors)
# log

push: top_object
schema ids from parent: [ref(0)]
my schema ids: [ref(0)]
additional: True
push: top_object.a
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(2)]
pop: top_object.a
my states: []
my schema ids: [ref(2)]
children schema_ids: []
verifiy node: schema2
validation errors: {'type': <NO ERROR>: valid}
push: top_object.b
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(3)]
additional: False
push: top_object.b.c
schema ids from parent: [{'c': ref(5), None: ref(-2)}]
my schema ids: [ref(5)]
pop: top_object.b.c
my states: []
my schema ids: [ref(5)]
children schema_ids: []
verifiy node: schema5
validation errors: {'type': <NO ERROR>: valid}
pop: top_object.b
my states: []
my schema ids: [ref(3)]
children schema_ids: [{'c': ref(5), None: ref(-2)}]
verifiy node: schema3
start verifying: [[    <NO ERROR>: valid
]]
all states: [[    <NO ERROR>: valid
]], state idx: 0
validate all: [F

In [7]:
# maximum stack depth verification
log = verify_json(data=data_file, max_depth=2)

target_errors = {
    'circuit_breaker': ValidationError(ErrorType.DEPTH_ERROR, {'depth': 2})
}
# assert_all_logs(log, target_errors)
# log

push: top_object
schema ids from parent: [ref(0)]
my schema ids: [ref(0)]
additional: True
push: top_object.a
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(2)]
pop: top_object.a
my states: []
my schema ids: [ref(2)]
children schema_ids: []
verifiy node: schema2
validation errors: {'type': <NO ERROR>: valid}
push: top_object.b
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(3)]
additional: False
push: top_object.b.c
schema ids from parent: [{'c': ref(5), None: ref(-2)}]
my schema ids: [ref(5)]
[invalid json item: path(circuit_breaker)
    <DEPTH ERROR>: maximum allowed depth exceeded: 2
]


In [8]:
# enumeration error verification
log = verify_json(data=data_enum_error_file)
target_errors = {
    'top_object.d[2]': ValidationError(ErrorType.BAD_VALUE, {'value': 5, 'rule': 'enum([3, 4])'}),
    'top_object.d': ValidationError(ErrorType.INCOMPLETE, {'value': 2}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'd'})
}

# assert_all_logs(log, target_errors)
# log

push: top_object
schema ids from parent: [ref(0)]
my schema ids: [ref(0)]
additional: True
push: top_object.a
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(2)]
pop: top_object.a
my states: []
my schema ids: [ref(2)]
children schema_ids: []
verifiy node: schema2
validation errors: {'type': <NO ERROR>: valid}
push: top_object.b
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(3)]
additional: False
push: top_object.b.c
schema ids from parent: [{'c': ref(5), None: ref(-2)}]
my schema ids: [ref(5)]
pop: top_object.b.c
my states: []
my schema ids: [ref(5)]
children schema_ids: []
verifiy node: schema5
validation errors: {'type': <NO ERROR>: valid}
pop: top_object.b
my states: []
my schema ids: [ref(3)]
children schema_ids: [{'c': ref(5), None: ref(-2)}]
verifiy node: schema3
start verifying: [[    <NO ERROR>: valid
]]
all states: [[    <NO ERROR>: valid
]], state idx: 0
validate all: [F

In [9]:
# unexpected json object verification
log = verify_json(data=data_no_schema_error_file)

target_errors = {
    'top_object.b.e': ValidationError(ErrorType.UNEXPECTED, {'value': 'extra'}),
    'top_object.b': ValidationError(ErrorType.INCOMPLETE, {'value': 'e'}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'b'})
}

# assert_all_logs(log, target_errors)
# log

push: top_object
schema ids from parent: [ref(0)]
my schema ids: [ref(0)]
additional: True
push: top_object.a
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(2)]
pop: top_object.a
my states: []
my schema ids: [ref(2)]
children schema_ids: []
verifiy node: schema2
validation errors: {'type': <NO ERROR>: valid}
push: top_object.b
schema ids from parent: [{'a': ref(2), 'b': ref(3), 'd': ref(6), None: ref(-1)}]
my schema ids: [ref(3)]
additional: False
push: top_object.b.c
schema ids from parent: [{'c': ref(5), None: ref(-2)}]
my schema ids: [ref(5)]
pop: top_object.b.c
my states: []
my schema ids: [ref(5)]
children schema_ids: []
verifiy node: schema5
validation errors: {'type': <NO ERROR>: valid}
push: top_object.b.e
schema ids from parent: [{'c': ref(5), None: ref(-2)}]
my schema ids: [ref(-2)]
pop: top_object.b.e
my states: []
my schema ids: [ref(-2)]
children schema_ids: []
verifiy node: schema-2
pop: top_object.b
my states: []
my s